# RPA reductions of Gibbs states

For a Gibbs state $\rho = e^{-{\bf K}}/Z$ of a large system, the reduced state of a small subsystem $A$ cannot be computed exactly. The default approximation of QALMA is a self-consistent mean field, which treats the environment as a product state $\rho_A=e^{-{\bf K}_{A,\mathrm{MF}}}/Z_{A,MF}$. The **RPA reduction** adds the contribution of the Gaussian fluctuations of the environment to the reduced generator:

$${\bf K}_A' = {\bf K}_{A,\mathrm{MF}} + {\bf K}_{A,\mathrm{RPA}} .$$

Since ${\bf K}_A'$ is Hermitian, $e^{-{\bf K}_A'}$ is a positive operator.

This notebook walks through the tools of the module `qalma.meanfield.rpa`:

1. static, dynamical and hybrid reductions of a pair of spins in a chain;
2. their dependence on the temperature;
3. the reduced generator and the reuse of the RPA data;
4. the `environment` option, and a system beyond exact diagonalization;
5. the bosonized reduced states of a fully connected model;
6. entropies and negativities of the bosonic RPA state;
7. systems with a broken continuous symmetry (zero modes).

The mathematical details are in the [RPA user guide](../user/rpa.rst).

**Note:** the RPA reduction is experimental: its interface may change in future releases.

In [ ]:
import time
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import qutip

from qalma import build_system
from qalma.meanfield import variational_quadratic_mfa
from qalma.meanfield.rpa import RPAGibbsDensityOperator
from qalma.operators.states import GibbsDensityOperator
from qalma.projections.symmetries import project_conserved_quantity

# Populations of the order of 1e-26 at low temperature make scipy.linalg.logm
# emit this (harmless) warning in the example of the last section.
warnings.filterwarnings("ignore", message="The logm input matrix may be nearly singular")

## 1. A pair of spins in a Heisenberg chain

We take a Heisenberg chain of 8 spins $1/2$ in a magnetic field, at inverse temperature $\beta = 0.8$, and look at the nearest-neighbour correlation $\langle S^z_0 S^z_1\rangle$ of the first two sites. The generator is $K = \beta (H - h S^z)$ with $h = 0.5$.

In [ ]:
system = build_system("chain lattice", "spin", L=6)
H = system.global_operator("Hamiltonian")
Sz = system.global_operator("Sz")

beta = 0.8
k = (beta * (H - 0.5 * Sz)).simplify()

sites = frozenset({"1[0]", "1[1]"})
corr = (system.site_operator("Sz", "1[0]") * system.site_operator("Sz", "1[1]")).simplify()

`RPAGibbsDensityOperator` is a Gibbs state whose `partial_trace` includes the RPA. Three reductions are available:

* `dynamic=False` (default): *static* correction of the generator. It is the cheapest one.
* `dynamic=True`: *dynamical* correction. The environment is bosonized in the eigenbasis of the local mean-field states, which is equivalent to summing all the Matsubara frequencies of the Hubbard-Stratonovich fluctuations.
* `hybrid=True`: the subsystem is kept exact and only the environment is treated in the RPA. It is non perturbative in the coupling between the subsystem and the environment, and it is intended for low temperatures.

Reductions that involve at most `max_exact_sites` sites are computed exactly (the default is `qalma.settings.MAXIMUM_GIBBS_EXACT_PARTIAL_TRACE`, 8 sites), so we pass `max_exact_sites=0` to see the effect of the approximation. A plain `GibbsDensityOperator` only provides the mean-field reduction, and, for 8 sites, its exact one; we use the latter as the reference.

In [ ]:
state = RPAGibbsDensityOperator(k)

values = {"exact": GibbsDensityOperator(k).partial_trace(sites).expect(corr)}
values["mean field"] = state.partial_trace(sites, rpa=False, max_exact_sites=0).expect(corr)
values["RPA, static"] = state.partial_trace(sites, max_exact_sites=0).expect(corr)
values["RPA, dynamic"] = state.partial_trace(sites, dynamic=True, max_exact_sites=0).expect(corr)
values["RPA, hybrid"] = state.partial_trace(
    sites, hybrid=True, max_exact_sites=0, hybrid_options={"n_gauss": 3}
).expect(corr)

for name, value in values.items():
    print(f"{name:14s} {value.real: .5f}")

This is a single example of a small system, not a benchmark: the static correction does not improve on the mean field here, while the dynamical and hybrid reductions do. The hybrid reduction diagonalizes exactly a Hamiltonian that contains the subsystem and a few effective bosons that represent the environment; its options are passed in `hybrid_options`, and its cost grows quickly with the number of effective bosons (`n_gauss`) and of Fock states (`n_fock`).

## 2. Dependence on the temperature

The three corrections of the generator can be compared along a range of temperatures. In this range the dynamical correction is the closest to the exact value, while the static one, which is perturbative, is already slightly worse than the mean field at $\beta = 0.5$ and clearly worse at lower temperatures.

In [ ]:
betas = [0.2, 0.5, 0.8, 1.2, 1.6, 2.0]
exact_scan, mf_scan, static_scan, dynamic_scan = [], [], [], []
for b in betas:
    k_b = (b * (H - 0.5 * Sz)).simplify()
    exact_scan.append(GibbsDensityOperator(k_b).partial_trace(sites).expect(corr).real)
    state_b = RPAGibbsDensityOperator(k_b)
    for result, options in (
        (mf_scan, {"rpa": False}),
        (static_scan, {}),
        (dynamic_scan, {"dynamic": True}),
    ):
        result.append(state_b.partial_trace(sites, max_exact_sites=0, **options).expect(corr).real)

exact_scan = np.array(exact_scan)
fig, ax = plt.subplots(figsize=(5, 3.5))
for label, data in (("mean field", mf_scan), ("RPA, static", static_scan), ("RPA, dynamic", dynamic_scan)):
    ax.plot(betas, np.abs(np.array(data) - exact_scan), "o-", label=label)
ax.set_yscale("log")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"error of $\langle S^z_0 S^z_1\rangle$")
ax.legend()
plt.show()

## 3. The reduced generator and the reuse of the RPA data

`reduced_generator` returns the approximate generator $K_A'$ of the subsystem. Since it is Hermitian, $e^{-K_A'}$ is positive for any of the reductions.

In [ ]:
block = tuple(sorted(sites))
for label, options in (("mean field", {"rpa": False}), ("static", {}), ("dynamic", {"dynamic": True})):
    generator = state.reduced_generator(sites, **options).to_qutip(block)
    rho = (-generator).expm()
    rho = rho / rho.tr()
    print(f"{label:10s} Hermitian: {generator.isherm}  smallest eigenvalue of the state: {min(rho.eigenenergies()):.2e}")

The part of the RPA that does not depend on the subsystem (the mean-field state, the coupling matrix, the local bases and the bosonic modes) is stored in `rpa_structure`, so it is computed only once when several subsystems are reduced.

In [ ]:
state_dyn = RPAGibbsDensityOperator(k, dynamic=True)
pairs = [frozenset({f"1[{i}]", f"1[{i + 1}]"}) for i in range(0, 5, 2)]
for pair in pairs:
    t0 = time.perf_counter()
    rho = state_dyn.partial_trace(pair, max_exact_sites=0)
    elapsed = time.perf_counter() - t0
    first, second = sorted(pair)
    local_corr = (system.site_operator("Sz", first) * system.site_operator("Sz", second)).simplify()
    print(f"{first}-{second}: <SzSz> = {rho.expect(local_corr).real: .5f}   ({elapsed:.2f} s)")

The first call includes the computation of the structure, which the following ones reuse (all the pairs give the same value because the chain is periodic and translation invariant). The default values of `dynamic`, `hybrid` and `hybrid_options` are those given to the constructor, and can be overridden in each call.

## 4. The environment, and larger systems

The static and dynamical corrections need the correlations of the environment. The option `environment` selects where they come from:

* `"isolated"` (default): the environment is bosonized with the subsystem removed, which needs a Bogoliubov transformation for each subsystem;
* `"global"`: the correlations are those of the bosonization of the whole system, computed once and shared by all the subsystems. It is somewhat less accurate, but its cost per subsystem grows much more slowly with the size of the system.

In [ ]:
for environment in ("isolated", "global"):
    state_env = RPAGibbsDensityOperator(k, dynamic=True, environment=environment)
    value = state_env.partial_trace(sites, max_exact_sites=0).expect(corr).real
    print(f"{environment:9s} {value: .5f}   (exact: {values['exact'].real: .5f})")

With the global environment, a chain of 40 spins is within reach. No exact reference is available for it, but the correlation is close to the one of the chain of 8 spins. The first reduction builds the bosonization of the whole system, which is the expensive part; the following ones reuse it.

In [ ]:
system_10 = build_system("chain lattice", "spin", L=10)
k_10 = (beta * (system_10.global_operator("Hamiltonian") - 0.5 * system_10.global_operator("Sz"))).simplify()
state_10 = RPAGibbsDensityOperator(k_10, dynamic=True, environment="global")

for first, second in (("1[8]", "1[9]"), ("1[7]", "1[9]")):
    pair = frozenset({first, second})
    local_corr = (system_10.site_operator("Sz", first) * system_10.site_operator("Sz", second)).simplify()
    t0 = time.perf_counter()
    value = state_10.partial_trace(pair).expect(local_corr).real
    mean_field = state_10.partial_trace(pair, rpa=False).expect(local_corr).real
    print(f"{first}-{second}: RPA {value: .5f}   mean field {mean_field: .5f}   ({time.perf_counter() - t0:.2f} s)")

## 5. Bosonized reduced states of a fully connected model

The Gaussian state of the full bosonization also gives the reduced states of one or two sites, with `bosonized_state`. The averages of products of operators of two sites follow from Wick's theorem, and the bosonization of the whole system is computed only once. This reduction is meant for models with many weak couplings (here, a fully connected XY model with a transverse field) and not for short-range chains, where its matrices can have large negative eigenvalues.

We compare the trace distance to the exact reduced state of a pair of spins for six spins at a low temperature.

In [ ]:
def fully_connected(system, field, couplings):
    # Transverse field plus an all-to-all coupling of strength 2 * J / n.
    names = list(system.sites)
    n_sites = len(names)

    def op(name, site):
        return system.site_operator(f"{name}@{site}")

    hamiltonian = field * sum(op("Sz", site) for site in names)
    for i, site_i in enumerate(names):
        for site_j in names[i + 1 :]:
            for name, value in couplings.items():
                hamiltonian = hamiltonian - (2.0 * value / n_sites) * (op(name, site_i) * op(name, site_j))
    return hamiltonian.simplify()


system_fc = build_system("chain lattice", "spin", L=6, J=1, local_S=0.5)
pair_fc = tuple(list(system_fc.sites)[:2])
k_fc = (fully_connected(system_fc, 1.5, {"Sx": 1.0, "Sy": 0.5}) / 0.3).simplify()


def normalized(reduced_state):
    rho = reduced_state.to_qutip(pair_fc)
    return rho / rho.tr()


exact_fc = normalized(GibbsDensityOperator(k_fc).partial_trace(frozenset(pair_fc)))
state_fc = RPAGibbsDensityOperator(k_fc, system_fc)
state_fc_global = RPAGibbsDensityOperator(k_fc, system_fc, dynamic=True, environment="global")

reduced = {
    "mean field": state_fc.partial_trace(frozenset(pair_fc), max_exact_sites=0, rpa=False),
    "RPA, dynamic": state_fc.partial_trace(frozenset(pair_fc), max_exact_sites=0, dynamic=True),
    "RPA, dynamic, global environment": state_fc_global.partial_trace(frozenset(pair_fc), max_exact_sites=0),
    "bosonized state": state_fc.bosonized_state(frozenset(pair_fc)),
}
for name, reduced_state in reduced.items():
    distance = 0.5 * (normalized(reduced_state) - exact_fc).norm("tr")
    print(f"{name:34s} trace distance to the exact state: {distance:.4f}")

## 6. Entropies and negativities of the bosonic RPA state

If the subsystem is also bosonized, the whole system is a quadratic bosonic Hamiltonian and its thermal state is Gaussian (a `qalma.gaussian.BosonicGaussianSystem`, available as `state.rpa_structure.boson_system()`). The reduced covariance matrix of any set of sites gives the entropy and the negativity of the bosonic RPA state: `rpa_entropy` and `rpa_negativity`.

As an example, take two spins coupled by an XY interaction in a transverse field, the model of Matera, Rossignoli and Canosa, Phys. Rev. A **82**, 052332. At low temperature, the RPA gives closed expressions for the entropy of one spin and for the negativity in terms of the average boson occupation $f$, and they do not depend on the value of the spin.

In [ ]:
def xy_pair(spin, field, beta, jx=1.0, jy=0.5):
    # Two spins with H = field * (Sz_a + Sz_b) - (1/spin) * (jx Sx_a Sx_b + jy Sy_a Sy_b).
    system_pair = build_system("chain lattice", "spin", L=2, J=1, local_S=spin)
    site_a, site_b = list(system_pair.sites)

    def op(name, site):
        return system_pair.site_operator(f"{name}@{site}")

    hamiltonian = field * (op("Sz", site_a) + op("Sz", site_b))
    for name, coupling in (("Sx", jx), ("Sy", jy)):
        if coupling:
            hamiltonian = hamiltonian - (1.0 / spin) * coupling * (op(name, site_a) * op(name, site_b))
    k_pair = (beta * hamiltonian).simplify()
    meanfield = variational_quadratic_mfa(k_pair.flat())
    return RPAGibbsDensityOperator(k_pair, system_pair, meanfield=meanfield), site_a, site_b


field, jx, jy = 1.2, 1.0, 0.5
# Average boson occupation of the RPA in the normal phase (field > jx), Eqs. (46), (51) and (62) of arXiv:1009.1750
w_0 = np.sqrt((field - jx) * (field - jy))
w_1 = np.sqrt((field + jx) * (field + jy))
omega = 0.5 * (w_0 + w_1)
f = 0.5 * (np.sqrt(1 + (field**2 - omega**2) / (w_0 * w_1)) - 1)
print(f"paper:       entropy {(1 + f) * np.log2(1 + f) - f * np.log2(f):.5f}   negativity {f + np.sqrt(f * (f + 1)):.5f}")

for spin in (0.5, 2.0):
    pair_state, site_a, site_b = xy_pair(spin, field, beta=60.0)
    entropy = pair_state.rpa_entropy(frozenset([site_a]))
    negativity = pair_state.rpa_negativity(frozenset([site_a]), frozenset([site_b]))
    print(f"spin {spin}:     entropy {entropy:.5f}   negativity {negativity:.5f}   zero modes: {pair_state.rpa_zero_modes()}")

These are quantities of the bosonic RPA state, not of the spin state, and they describe the quantum fluctuations around the mean-field state: the *static* modes (the diagonal operators of each site) are classical fluctuations and are left out.

## 7. Broken continuous symmetries

If the mean-field state breaks a continuous symmetry, the bosonic Hamiltonian has *zero modes* (Goldstone modes). They are removed from the Gaussian state, which then is not a physical state of the local modes, and `rpa_entropy` and `rpa_negativity` raise a `ValueError`. In this case the entanglement is computed from the hybrid reduced states, restoring the symmetry with the `symmetry_projections` of the state (here, `project_conserved_quantity` for the $U(1)$ symmetry of the XXZ model). The orientation of the order parameter is then integrated exactly, and the RPA is applied only to the intrinsic variables.

We take an isotropic XY pair ($J_x = J_y$) in a weak field at low temperature, whose exact state has one bit of entropy per spin.

In [ ]:
pair_state, site_a, site_b = xy_pair(0.5, 0.5, beta=40.0, jy=1.0)
print("zero modes:", pair_state.rpa_zero_modes())
try:
    pair_state.rpa_entropy(frozenset([site_a]))
except ValueError as error:
    print("ValueError:", str(error)[:110], "...")

In [ ]:
projection = partial(project_conserved_quantity, op_name="Sz")
symmetric_state = RPAGibbsDensityOperator(
    pair_state.k,
    pair_state.system,
    meanfield=pair_state.meanfield,
    symmetry_projections=[projection],
)


def entropy_of_one_spin(reduced_state):
    rho = reduced_state.to_qutip((site_a,))
    return qutip.entropy_vn(rho / rho.tr(), base=2)


one_site = frozenset([site_a])
exact_state = symmetric_state.to_qutip_operator().partial_trace(one_site)
print(f"exact       {entropy_of_one_spin(exact_state):.3f} bits")
print(f"mean field  {entropy_of_one_spin(symmetric_state.partial_trace(one_site, max_exact_sites=0, rpa=False)):.3f} bits")
print(f"hybrid      {entropy_of_one_spin(symmetric_state.partial_trace(one_site, max_exact_sites=0, hybrid=True)):.3f} bits")

The mean-field state of the spin is a product state with a fixed orientation and gives a smaller entropy, while the hybrid reduction with the symmetry projection is close to the exact one.

## Where to go next

* [RPA user guide](../user/rpa.rst): formulation, limitations and the options of every reduction.
* [Variational mean field](../user/meanfield_variational.rst): the mean-field states that the RPA starts from.
* [API reference](../api/meanfield.rst): `qalma.meanfield.rpa` and `qalma.gaussian`.